### Imports and Setup

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime
import uuid

silver_run_id = str(uuid.uuid4())
print("Silver Run ID:", silver_run_id)

### Create silver control table

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS olist_ecommerce.silver.processing_control (
    layer STRING,
    entity_name STRING,
    last_processed_bronze_run_id STRING,
    last_processed_bronze_ingested_at TIMESTAMP,
    rows_merged BIGINT,
    rows_rejected BIGINT,
    run_status STRING,
    silver_run_id STRING,
    updated_at TIMESTAMP
)
USING DELTA
""")

### Create rejected records table

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS olist_ecommerce.silver.rejected_records (
    entity_name STRING,
    rejection_reason STRING,
    rejected_record STRING,
    bronze_run_id STRING,
    silver_run_id STRING,
    rejected_at TIMESTAMP
)
USING DELTA
""")

### Create helper functions

In [0]:
def get_last_processed_bronze_ingested_at(entity_name: str):
    ctrl_df = (
        spark.table("olist_ecommerce.silver.processing_control")
        .filter(
            (F.col("layer") == "silver") &
            (F.col("entity_name") == entity_name) &
            (F.col("run_status") == "SUCCESS")
        )
        .orderBy(F.col("updated_at").desc())
        .limit(1)
    )

    rows = ctrl_df.collect()

    if not rows:
        return None

    return rows[0]["last_processed_bronze_ingested_at"]

In [0]:
def get_incremental_bronze(bronze_table: str, entity_name: str):
    last_ts = get_last_processed_bronze_ingested_at(entity_name)

    bronze_df = spark.table(bronze_table)

    if last_ts is None:
        print(f"First Silver run for {entity_name}. Processing all Bronze rows.")
        return bronze_df

    print(f"Processing only Bronze rows after: {last_ts}")

    return bronze_df.filter(F.col("bronze_ingested_at") > F.lit(last_ts))

In [0]:
def upsert_to_silver(df_source, target_table: str, join_condition: str):
    if spark.catalog.tableExists(target_table):
        delta_tbl = DeltaTable.forName(spark, target_table)

        (
            delta_tbl.alias("target")
            .merge(df_source.alias("source"), join_condition)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        (
            df_source.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", True)
            .saveAsTable(target_table)
        )

In [0]:
def upsert_silver_control(
    entity_name,
    last_bronze_run_id,
    last_bronze_ingested_at,
    rows_merged,
    rows_rejected
):
    control_df = spark.createDataFrame(
        [(
            "silver",
            entity_name,
            last_bronze_run_id,
            last_bronze_ingested_at,
            int(rows_merged),
            int(rows_rejected),
            "SUCCESS",
            silver_run_id,
            datetime.utcnow()
        )],
        """
        layer STRING,
        entity_name STRING,
        last_processed_bronze_run_id STRING,
        last_processed_bronze_ingested_at TIMESTAMP,
        rows_merged BIGINT,
        rows_rejected BIGINT,
        run_status STRING,
        silver_run_id STRING,
        updated_at TIMESTAMP
        """
    )

    delta_tbl = DeltaTable.forName(
        spark,
        "olist_ecommerce.silver.processing_control"
    )

    (
        delta_tbl.alias("target")
        .merge(
            control_df.alias("source"),
            "target.layer = source.layer AND target.entity_name = source.entity_name"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
def write_rejected_records(df_bad, entity_name, reason_col):
    if df_bad.count() == 0:
        return

    rejected_df = (
        df_bad
        .withColumn("entity_name", F.lit(entity_name))
        .withColumn("rejection_reason", F.col(reason_col))
        .withColumn("rejected_record", F.to_json(F.struct("*")))
        .withColumn("silver_run_id", F.lit(silver_run_id))
        .withColumn("rejected_at", F.current_timestamp())
        .select(
            "entity_name",
            "rejection_reason",
            "rejected_record",
            "bronze_run_id",
            "silver_run_id",
            "rejected_at"
        )
    )

    (
        rejected_df.write
        .format("delta")
        .mode("append")
        .saveAsTable("olist_ecommerce.silver.rejected_records")
    )

## Cleaning Functions

### Customers

In [0]:
def transform_customers(df):
    window_spec = Window.partitionBy("customer_id").orderBy(F.col("bronze_ingested_at").desc())

    df_clean = (
        df
        .withColumn("customer_city", F.upper(F.trim("customer_city")))
        .withColumn("customer_state", F.upper(F.trim("customer_state")))
        .withColumn("customer_zip_code_prefix", F.col("customer_zip_code_prefix").cast("int"))
        .withColumn("dq_issue",
            F.when(F.col("customer_id").isNull(), "customer_id_missing")
             .when(F.col("customer_unique_id").isNull(), "customer_unique_id_missing")
             .when(F.col("customer_zip_code_prefix").isNull(), "zip_missing")
             .otherwise("No Issues")
        )
    )

    good_df = df_clean.filter(F.col("dq_issue") == "No Issues")
    bad_df = df_clean.filter(F.col("dq_issue") != "No Issues")

    good_df = (
        good_df
        .withColumn("rn", F.row_number().over(window_spec))
        .filter(F.col("rn") == 1)
        .drop("rn", "dq_issue")
        .withColumn("silver_processed_at", F.current_timestamp())
        .withColumn("silver_run_id", F.lit(silver_run_id))
    )

    return good_df, bad_df

### Sellers

In [0]:
def transform_sellers(df):
    window_spec = Window.partitionBy("seller_id").orderBy(F.col("bronze_ingested_at").desc())

    df_clean = (
        df
        .withColumn("seller_city", F.upper(F.trim("seller_city")))
        .withColumn("seller_state", F.upper(F.trim("seller_state")))
        .withColumn("seller_zip_code_prefix", F.col("seller_zip_code_prefix").cast("int"))
        .withColumn("dq_issue",
            F.when(F.col("seller_id").isNull(), "seller_id_missing")
             .when(F.col("seller_zip_code_prefix").isNull(), "seller_zip_missing")
             .otherwise("No Issues")
        )
    )

    good_df = df_clean.filter(F.col("dq_issue") == "No Issues")
    bad_df = df_clean.filter(F.col("dq_issue") != "No Issues")

    good_df = (
        good_df
        .withColumn("rn", F.row_number().over(window_spec))
        .filter(F.col("rn") == 1)
        .drop("rn", "dq_issue")
        .withColumn("silver_processed_at", F.current_timestamp())
        .withColumn("silver_run_id", F.lit(silver_run_id))
    )

    return good_df, bad_df

### Orders

In [0]:
def transform_orders(df):
    window_spec = Window.partitionBy("order_id").orderBy(F.col("bronze_ingested_at").desc())

    valid_status = [
        "DELIVERED",
        "SHIPPED",
        "CANCELED",
        "INVOICED",
        "PROCESSING",
        "UNAVAILABLE",
        "APPROVED",
        "CREATED"
    ]

    df_clean = (
        df
        .withColumn("order_status", F.upper(F.trim("order_status")))
        .withColumn("order_purchase_timestamp", F.to_timestamp("order_purchase_timestamp"))
        .withColumn("order_approved_at", F.to_timestamp("order_approved_at"))
        .withColumn("order_delivered_carrier_date", F.to_timestamp("order_delivered_carrier_date"))
        .withColumn("order_delivered_customer_date", F.to_timestamp("order_delivered_customer_date"))
        .withColumn("order_estimated_delivery_date", F.to_timestamp("order_estimated_delivery_date"))
        .withColumn(
            "delivery_days",
            F.datediff("order_delivered_customer_date", "order_purchase_timestamp")
        )
        .withColumn(
            "delay_days",
            F.datediff("order_delivered_customer_date", "order_estimated_delivery_date")
        )
        .withColumn(
            "on_time_flag",
            F.when(F.col("delay_days") <= 0, 1).otherwise(0)
        )
        .withColumn("dq_issue",
            F.when(F.col("order_id").isNull(), "order_id_missing")
             .when(F.col("customer_id").isNull(), "customer_id_missing")
             .when(~F.col("order_status").isin(valid_status), "invalid_order_status")
             .when(
                 (F.col("order_delivered_customer_date").isNotNull()) &
                 (F.col("order_purchase_timestamp") > F.col("order_delivered_customer_date")),
                 "purchase_after_delivery"
             )
             .otherwise("No Issues")
        )
    )

    good_df = df_clean.filter(F.col("dq_issue") == "No Issues")
    bad_df = df_clean.filter(F.col("dq_issue") != "No Issues")

    good_df = (
        good_df
        .withColumn("rn", F.row_number().over(window_spec))
        .filter(F.col("rn") == 1)
        .drop("rn", "dq_issue")
        .withColumn("silver_processed_at", F.current_timestamp())
        .withColumn("silver_run_id", F.lit(silver_run_id))
    )

    return good_df, bad_df

### Order Items

In [0]:
def transform_order_items(df):
    window_spec = Window.partitionBy("order_id", "order_item_id").orderBy(F.col("bronze_ingested_at").desc())

    df_clean = (
        df
        .withColumn("order_item_id", F.col("order_item_id").cast("int"))
        .withColumn("shipping_limit_date", F.to_timestamp("shipping_limit_date"))
        .withColumn("price", F.col("price").cast("double"))
        .withColumn("freight_value", F.col("freight_value").cast("double"))
        .withColumn("total_item_amount", F.col("price") + F.col("freight_value"))
        .withColumn(
            "dq_issue",
            F.when(F.col("order_id").isNull(), "order_id_missing")
             .when(F.col("order_item_id").isNull(), "order_item_id_missing")
             .when(F.col("product_id").isNull(), "product_id_missing")
             .when(F.col("seller_id").isNull(), "seller_id_missing")
             .when(F.col("price").isNull() | (F.col("price") <= 0), "invalid_price")
             .when(F.col("freight_value").isNull() | (F.col("freight_value") < 0), "invalid_freight")
             .otherwise("No Issues")
        )
    )

    good_df = (
        df_clean
        .filter(F.col("dq_issue") == "No Issues")
        .withColumn("rn", F.row_number().over(window_spec))
        .filter(F.col("rn") == 1)
        .drop("rn", "dq_issue")
        .withColumn("silver_processed_at", F.current_timestamp())
        .withColumn("silver_run_id", F.lit(silver_run_id))
    )

    bad_df = df_clean.filter(F.col("dq_issue") != "No Issues")

    return good_df, bad_df

### Order Payments

In [0]:
def transform_order_payments(df):
    window_spec = Window.partitionBy("order_id", "payment_sequential").orderBy(F.col("bronze_ingested_at").desc())

    valid_payment_types = [
        "CREDIT_CARD",
        "BOLETO",
        "VOUCHER",
        "DEBIT_CARD",
        "NOT_DEFINED"
    ]

    df_clean = (
        df
        .withColumn("payment_type", F.upper(F.trim("payment_type")))
        .withColumn("payment_sequential", F.col("payment_sequential").cast("int"))
        .withColumn("payment_installments", F.col("payment_installments").cast("int"))
        .withColumn("payment_value", F.col("payment_value").cast("double"))
        .withColumn("dq_issue",
            F.when(F.col("order_id").isNull(), "order_id_missing")
             .when(F.col("payment_sequential").isNull(), "payment_sequence_missing")
             .when(~F.col("payment_type").isin(valid_payment_types), "invalid_payment_type")
             .when(F.col("payment_value").isNull() | (F.col("payment_value") <= 0), "invalid_payment_value")
             .when(F.col("payment_installments").isNull() | (F.col("payment_installments") < 0), "invalid_installments")
             .otherwise("No Issues")
        )
    )

    good_df = df_clean.filter(F.col("dq_issue") == "No Issues")
    bad_df = df_clean.filter(F.col("dq_issue") != "No Issues")

    good_df = (
        good_df
        .withColumn("rn", F.row_number().over(window_spec))
        .filter(F.col("rn") == 1)
        .drop("rn", "dq_issue")
        .withColumn("silver_processed_at", F.current_timestamp())
        .withColumn("silver_run_id", F.lit(silver_run_id))
    )

    return good_df, bad_df

### Products

In [0]:
def transform_products(df):
    window_spec = Window.partitionBy("product_id").orderBy(F.col("bronze_ingested_at").desc())

    df_clean = (
        df
        .withColumn("product_category_name", F.lower(F.trim("product_category_name")))
        .withColumn("product_name_lenght", F.expr("try_cast(product_name_lenght as int)"))
        .withColumn("product_description_lenght", F.expr("try_cast(product_description_lenght as int)"))
        .withColumn("product_photos_qty", F.expr("try_cast(product_photos_qty as int)"))
        .withColumn("product_weight_g", F.expr("try_cast(product_weight_g as double)"))
        .withColumn("product_length_cm", F.expr("try_cast(product_length_cm as double)"))
        .withColumn("product_height_cm", F.expr("try_cast(product_height_cm as double)"))
        .withColumn("product_width_cm", F.expr("try_cast(product_width_cm as double)"))
        .withColumn("product_volume_cm3", F.col("product_length_cm") * F.col("product_height_cm") * F.col("product_width_cm"))
        .withColumn(
            "weight_category",
            F.when(F.col("product_weight_g") <= 500, "LIGHT")
             .when(F.col("product_weight_g") <= 2000, "MEDIUM")
             .otherwise("HEAVY")
        )
        .withColumn(
            "dq_issue",
            F.when(F.col("product_id").isNull(), "product_id_missing")
             .when(F.col("product_weight_g").isNull() | (F.col("product_weight_g") <= 0), "invalid_weight")
             .when(F.col("product_length_cm").isNull() | (F.col("product_length_cm") <= 0), "invalid_length")
             .when(F.col("product_height_cm").isNull() | (F.col("product_height_cm") <= 0), "invalid_height")
             .when(F.col("product_width_cm").isNull() | (F.col("product_width_cm") <= 0), "invalid_width")
             .otherwise("No Issues")
        )
    )

    good_df = (
        df_clean
        .filter(F.col("dq_issue") == "No Issues")
        .withColumn("rn", F.row_number().over(window_spec))
        .filter(F.col("rn") == 1)
        .drop("rn", "dq_issue")
        .withColumn("silver_processed_at", F.current_timestamp())
        .withColumn("silver_run_id", F.lit(silver_run_id))
    )

    bad_df = df_clean.filter(F.col("dq_issue") != "No Issues")

    return good_df, bad_df

### Order Reviews

In [0]:
def transform_order_reviews(df):
    window_spec = Window.partitionBy("review_id").orderBy(F.col("bronze_ingested_at").desc())

    df_clean = (
        df
        .withColumn("review_score", F.expr("try_cast(review_score as int)"))
        .withColumn("review_creation_date", F.expr("try_cast(review_creation_date as timestamp)"))
        .withColumn("review_answer_timestamp", F.expr("try_cast(review_answer_timestamp as timestamp)"))
        .withColumn(
            "review_sentiment",
            F.when(F.col("review_score") >= 4, "POSITIVE")
             .when(F.col("review_score") == 3, "NEUTRAL")
             .otherwise("NEGATIVE")
        )
        .withColumn("review_comment_length", F.length(F.col("review_comment_message")))
        .withColumn("dq_issue",
            F.when(F.col("review_id").isNull(), "review_id_missing")
             .when(F.col("order_id").isNull(), "order_id_missing")
             .when(~F.col("review_score").between(1, 5), "invalid_review_score")
             .otherwise("No Issues")
        )
    )

    good_df = df_clean.filter(F.col("dq_issue") == "No Issues")
    bad_df = df_clean.filter(F.col("dq_issue") != "No Issues")

    good_df = (
        good_df
        .withColumn("rn", F.row_number().over(window_spec))
        .filter(F.col("rn") == 1)
        .drop("rn", "dq_issue")
        .withColumn("silver_processed_at", F.current_timestamp())
        .withColumn("silver_run_id", F.lit(silver_run_id))
    )

    return good_df, bad_df

### Geolocation

In [0]:
def transform_geolocation(df):
    df_clean = (
        df
        .withColumn("geolocation_zip_code_prefix", F.col("geolocation_zip_code_prefix").cast("int"))
        .withColumn("geolocation_lat", F.col("geolocation_lat").cast("double"))
        .withColumn("geolocation_lng", F.col("geolocation_lng").cast("double"))
        .withColumn("geolocation_city", F.upper(F.trim("geolocation_city")))
        .withColumn("geolocation_state", F.upper(F.trim("geolocation_state")))
        .withColumn("dq_issue",
            F.when(F.col("geolocation_zip_code_prefix").isNull(), "zip_missing")
             .when(~F.col("geolocation_lat").between(-90, 90), "invalid_latitude")
             .when(~F.col("geolocation_lng").between(-180, 180), "invalid_longitude")
             .otherwise("No Issues")
        )
    )

    good_df = (
        df_clean
        .filter(F.col("dq_issue") == "No Issues")
        .dropDuplicates(["geolocation_zip_code_prefix", "geolocation_city", "geolocation_state"])
        .drop("dq_issue")
        .withColumn("silver_processed_at", F.current_timestamp())
        .withColumn("silver_run_id", F.lit(silver_run_id))
    )

    bad_df = df_clean.filter(F.col("dq_issue") != "No Issues")

    return good_df, bad_df

### Product category name translation

In [0]:
def transform_product_category_translation(df):
    df_clean = (
        df
        .withColumn("product_category_name", F.lower(F.trim("product_category_name")))
        .withColumn("product_category_name_english", F.upper(F.trim("product_category_name_english")))
        .withColumn("dq_issue",
            F.when(F.col("product_category_name").isNull(), "category_name_missing")
             .when(F.col("product_category_name_english").isNull(), "category_translation_missing")
             .otherwise("No Issues")
        )
    )

    good_df = (
        df_clean
        .filter(F.col("dq_issue") == "No Issues")
        .dropDuplicates(["product_category_name"])
        .drop("dq_issue")
        .withColumn("silver_processed_at", F.current_timestamp())
        .withColumn("silver_run_id", F.lit(silver_run_id))
    )

    bad_df = df_clean.filter(F.col("dq_issue") != "No Issues")

    return good_df, bad_df

### Silver Configuration

In [0]:
silver_config = {
    "customers": {
        "bronze_table": "olist_ecommerce.bronze.customers_raw",
        "silver_table": "olist_ecommerce.silver.customers_clean",
        "transform_func": transform_customers,
        "join_condition": "target.customer_id = source.customer_id"
    },
    "sellers": {
        "bronze_table": "olist_ecommerce.bronze.sellers_raw",
        "silver_table": "olist_ecommerce.silver.sellers_clean",
        "transform_func": transform_sellers,
        "join_condition": "target.seller_id = source.seller_id"
    },
    "orders": {
        "bronze_table": "olist_ecommerce.bronze.orders_raw",
        "silver_table": "olist_ecommerce.silver.orders_clean",
        "transform_func": transform_orders,
        "join_condition": "target.order_id = source.order_id"
    },
    "order_items": {
        "bronze_table": "olist_ecommerce.bronze.order_items_raw",
        "silver_table": "olist_ecommerce.silver.order_items_clean",
        "transform_func": transform_order_items,
        "join_condition": "target.order_id = source.order_id AND target.order_item_id = source.order_item_id"
    },
    "order_payments": {
        "bronze_table": "olist_ecommerce.bronze.order_payments_raw",
        "silver_table": "olist_ecommerce.silver.order_payments_clean",
        "transform_func": transform_order_payments,
        "join_condition": "target.order_id = source.order_id AND target.payment_sequential = source.payment_sequential"
    },
    "products": {
        "bronze_table": "olist_ecommerce.bronze.products_raw",
        "silver_table": "olist_ecommerce.silver.products_clean",
        "transform_func": transform_products,
        "join_condition": "target.product_id = source.product_id"
    },
    "order_reviews": {
        "bronze_table": "olist_ecommerce.bronze.order_reviews_raw",
        "silver_table": "olist_ecommerce.silver.order_reviews_clean",
        "transform_func": transform_order_reviews,
        "join_condition": "target.review_id = source.review_id"
    },
    "geolocation": {
        "bronze_table": "olist_ecommerce.bronze.geolocation_raw",
        "silver_table": "olist_ecommerce.silver.geolocation_clean",
        "transform_func": transform_geolocation,
        "join_condition": "target.geolocation_zip_code_prefix = source.geolocation_zip_code_prefix AND target.geolocation_city = source.geolocation_city AND target.geolocation_state = source.geolocation_state"
    },
    "product_category_translation": {
        "bronze_table": "olist_ecommerce.bronze.product_category_name_translation_raw",
        "silver_table": "olist_ecommerce.silver.product_category_translation_clean",
        "transform_func": transform_product_category_translation,
        "join_condition": "target.product_category_name = source.product_category_name"
    }
}

### Silver Loop

In [0]:
for entity_name, cfg in silver_config.items():
    print(f"Processing Silver entity: {entity_name}")

    if not spark.catalog.tableExists(cfg["bronze_table"]):
        print(f"Bronze table {cfg['bronze_table']} does not exist. Skipping {entity_name}.")
        continue

    bronze_df = get_incremental_bronze(
        cfg["bronze_table"],
        entity_name
    )

    rows_in_bronze = bronze_df.count()

    if rows_in_bronze == 0:
        print(f"No new Bronze rows for {entity_name}")
        continue

    good_df, bad_df = cfg["transform_func"](bronze_df)

    rows_good = good_df.count()
    rows_bad = bad_df.count()

    if rows_bad > 0:
        write_rejected_records(bad_df, entity_name, "dq_issue")

    if rows_good > 0:
        upsert_to_silver(
            good_df,
            cfg["silver_table"],
            cfg["join_condition"]
        )

    max_bronze_ingested_at = bronze_df.agg(
        F.max("bronze_ingested_at").alias("max_ts")
    ).collect()[0]["max_ts"]

    latest_bronze_run_id = (
        bronze_df
        .orderBy(F.col("bronze_ingested_at").desc())
        .select("bronze_run_id")
        .first()["bronze_run_id"]
    )

    upsert_silver_control(
        entity_name,
        latest_bronze_run_id,
        max_bronze_ingested_at,
        rows_good,
        rows_bad
    )

    print(f"Completed {entity_name}: good={rows_good}, rejected={rows_bad}")

### Validating Silver tables

In [0]:
display(spark.sql("""
SELECT *
FROM olist_ecommerce.silver.processing_control
ORDER BY updated_at DESC
"""))

In [0]:
display(spark.sql("""
SELECT entity_name, rejection_reason, COUNT(*) AS rejected_count
FROM olist_ecommerce.silver.rejected_records
GROUP BY entity_name, rejection_reason
ORDER BY entity_name, rejected_count DESC
"""))

In [0]:
for entity_name, cfg in silver_config.items():
    count_val = spark.table(cfg["silver_table"]).count()
    print(f"{entity_name}: {count_val}")

In [0]:
for entity_name, cfg in silver_config.items():
    table_name = cfg["silver_table"]
    print(f"Optimizing {table_name}")
    spark.sql(f"OPTIMIZE {table_name}")

In [0]:
spark.sql("OPTIMIZE olist_ecommerce.silver.orders_clean ZORDER BY (order_id)")
spark.sql("OPTIMIZE olist_ecommerce.silver.order_items_clean ZORDER BY (order_id, product_id)")
spark.sql("OPTIMIZE olist_ecommerce.silver.order_payments_clean ZORDER BY (order_id)")
spark.sql("OPTIMIZE olist_ecommerce.silver.products_clean ZORDER BY (product_id)")

In [0]:
silver_base_path = "abfss://silver@olistecommercesc.dfs.core.windows.net/"

In [0]:
silver_tables = [
    "customers_clean",
    "sellers_clean",
    "orders_clean",
    "order_items_clean",
    "order_payments_clean",
    "products_clean",
    "order_reviews_clean",
    "geolocation_clean",
    "product_category_translation_clean"
]

for table in silver_tables:
    source_table = f"olist_ecommerce.silver.{table}"
    target_path = f"{silver_base_path}{table}"

    df = spark.table(source_table)

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(target_path)
    )

    print(f"Written {source_table} to {target_path}")